In [1]:
import pandas as pd
import os 
import numpy as np
import geopandas as gpd
from glob import glob

# Set option to display all rows (no truncation)
pd.set_option('display.max_rows', None)

# Set option to display all columns (no truncation)
pd.set_option('display.max_columns', None)

/usr/local/lib/python3.8/dist-packages/geopandas/_compat.py:123: UserWarning: The Shapely GEOS version (3.11.2-CAPI-1.17.2) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(
/tmp/tmp.0EgCjqR4Xs/ipykernel_3315385/761733327.py:4: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_

In [2]:
d_readin = gpd.read_file("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events/ncei_ny_events_clean.gpkg")

d_readin.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")

# may take ~40 seconds

In [3]:
# Explore what warnings there are (no need to run)

print(np.unique(d_readin["EVENT_TYPE"]))

eventsofint = ['Blizzard', 'Heavy Rain', 'Heavy Snow', 'Lake-Effect Snow', 'Winter Storm' ,'Winter Weather']

devents = d_readin[d_readin["EVENT_TYPE"].isin(eventsofint)]

# print(len(devents)) # note that this isn't the number of instances to run for inference. Because 1) events elements are a range of time, so need to run multiple 5-min instances within one 'event', so this makes *more* instances, 2) If there an event that spans multiple regions, there is likely overlapping times of the warning. These will end up being duplicate because when we prepare the inference instances to run (below) we run the entire state for every time where there was an active snow squall warning, regardless of where the warning was (cleaner to maintain consistency in inference runs by running every instance statewide)

display(devents.groupby(["EVENT_TYPE"]).count())

['Astronomical Low Tide' 'Blizzard' 'Coastal Flood' 'Cold/Wind Chill'
 'Debris Flow' 'Drought' 'Excessive Heat' 'Extreme Cold/Wind Chill'
 'Flash Flood' 'Flood' 'Frost/Freeze' 'Funnel Cloud' 'Hail' 'Heat'
 'Heavy Rain' 'Heavy Snow' 'High Wind' 'Ice Storm' 'Lake-Effect Snow'
 'Lakeshore Flood' 'Lightning' 'Rip Current' 'Strong Wind'
 'Thunderstorm Wind' 'Tornado' 'Wildfire' 'Winter Storm' 'Winter Weather']


,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration
EVENT_TYPE,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Blizzard,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,14,7,14,14,14,0,0,14,14,14,14,14
Heavy Rain,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,14,6,6,14,0,0,0,0,0,0,0,0,0,0,0,14,14,14,14,14,14,14,14,14,14,14,11,14,14,0,14,14,14,14,14,14,14
Heavy Snow,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,185,148,148,185,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,185,157,185,185,185,0,0,185,185,185,185,185
Lake-Effect Snow,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,211,187,187,211,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,211,57,211,211,211,0,0,211,211,211,211,211
Winter Storm,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,437,311,311,437,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,437,177,437,437,437,0,0,437,437,437,437,437
Winter Weather,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,899,341,341,899,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,899,476,899,899,899,0,0,899,899,899,899,899


# Grab events to run
see reference: /home/csutter/DRIVE-clean/NWS_warnings/notebooks/nws_dataset_analysis.ipynb for more info

Important notes and tracking 
- See "HERE!!" comments below, there are many now that we adjust logic depending on event
- DONE: Blizzard 
    - On 1/23 ran the remaining blizzard events from NCEI that hadnt already been ran in NWS warnings, using the same 5-min logic and the same 15 min buffer that I did from the NWS code. Noting this, bc we changed the code since (see bullet below). Note we have some cases (set35) where there are no images thus no model runs. 
- DONE: Lake-Effect Snow. 
    - For lake effect, severe snow, there are far too many events of too long length, to run ALL lake effect events for ALL years for ALL 5 mins during duration. Solution, run instances: 1) every 30 min and 2) by years at a time 
    - DONE: On 1/23, ran "Lake-Effect Snow", "2025", .dt.ceil("h"), .dt.floor("30min"), freq="30min"
        - (starting w/ 2025, logic here bc i didnt label images from 2025 so we have some justification for that.)
    - On 1/25: run Lake-Effect for 2024 w/ 30 min ceil and freq (made new savedout events_ofint dataset w 2024 and 2025 togethr)
- On 1/25, ran "Heavy Snow"
    - on 1/25: these events have an avg duration of 1.3 days... so running every 5 min not a great choice, chose to do every 30 min like for lakeeffect. 2025 only has 2 events. Extend to 2024 (don't have too much labeled data for that year, so that's decently consistent w the logic for lakeeffect above)
- On 1/25 generally choosing to prioritize gather more events over gathering more frequency within smaller number of events
    - If i want my code to run for 5 days, I can aim for 2000 instances per run...
    - EVERY 15 min for 2024/2025 -- DECIDED THIS FOR AMS WEEK. Started jobs 1/25 around 2pm. Will take 4 ish days to run them at most. 
        - Lake-effect every 15 min is 2000 - 4 days
        - Heavy snow every 15 min is 1000 - 2 days
        - Heavy rain is tiny - SKIP BUT NEED TO RUN ONCE ONE IS DONE! *
        - Blizzard is done 
        - Winter Storm is 2200 - 4-5 days
        - Winter Weather is 3300 - 6 ish days
    - EVERY 30 min (roughly half of ^...) 
        - Lake-effect every 30 min for 2024/2025 is 567 obs -- 28h
        - Heavy snow is 430 - 21h
        - Heavy rain is tiny - fast
        - Blizzard is done 
        - Winter Storm is 1000 - 2d
        - Winter Weather 1300 - 2.7d


    



In [65]:
# grab snow squall data (or whatever event of interest is)
d_event_subset = d_readin[d_readin["EVENT_TYPE"]=='Heavy Rain'] # HERE!!
print(len(d_event_subset))

#TOO MANY EVENTS! Limit it to 2023 and 2024
d_event_subset = d_event_subset[d_event_subset["YEAR"].isin([2024,2025])]  # HERE!!

# just for reference
print(len(d_event_subset))

print(np.mean(d_event_subset["duration"]))

14
12
0 days 03:47:10


In [55]:
print(len(d_event_subset))

458


In [66]:
# Grab start and end times, for which we'll run all datetimes in between
# Will grab start time, end time, and list every 5-min increment in between
# Will also run 15 mins before and after squall starts to capture the prior and post conditions
# Round to 0005, 0010, etc, 5 min increments. Why? B/c 1) we only snapshot images in 5-min increments anyway, and while they won't be exactly on the even 5 mins, if we consistently run inference runs with ever 5 mins, it means we'll capture all instances (e.g. if we started allowing "off" times like 11 min rather than 10, that may be the same "image instance" for 1000 cams -- this is a waste of inference run). This allows us to keep track seamlessly across lots of different case study inference runs, exactly which datetimes have and have not been ran already
# Note: we run every instance statewide, just for consistency/ease of run (see note above about seamless tracking across case studies). I'm not parsing certain regions based on those that had the squall in region. 

d_event_subset["start_round"] = d_event_subset["BEGIN_UTC"].dt.ceil("15min") #HERE!!
# For events where we want to limit to running 30 mins, to keep things organized, initiate this rounding to the nearest hour AFTER the event began
# prior version for 5 min: .dt.round("5min") # DEPRECATE this for consistency, should really always just ceil, and then rely on the "buffer" if want to capture safe surrounding times.
# for rounding to 30 min .dt.ceil("30min"), or .dt.ceil("5min")
d_event_subset["end_round"] = d_event_subset["END_UTC"].dt.floor("15min") # HERE!!
# To Round end time either top or bottom of hour that is less than the actual end time (bc want this last observation to actually capture the event duration, not just blind rounding which may lead to after)
# .dt.round("5min") # Similar to note above for ceil, will DEPRECATE this
# .dt.floor("30min")


# HERE!! Comment out -/+ if dont want a buffer
# DEPRECATING THE USE OF BUFFER TIMES. For now my thinking is that this only really matters for short lived squall events, and that data is only in NWS warnings and we did use 15 min buffer for that. 
d_event_subset["start_buffer"] = d_event_subset["start_round"] #$- pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] #+ pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="15min"),  # HERE!! Adjust if 5 min too frequent if an event is quite long
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)

print(len(d_event_subset))



12


In [43]:
print(len(d_event_subset))

136


In [67]:
# SAVE OUT THIS CLEANED DATASET FOR WHEN DOING ANALYSIS/MODELING
# IMPORTANT: may want to skip this step initially while you see how many unique dates will ened to be ran based on the logic (and may tweak the logic we set accordingly based on the number of expected runs)... so keep running the code after this cell to make sure the # of runs matches what we'd want to run before saving this out. 
# Set proper naming for the type of subsetting logic - must include 1) event name 2) any year subsetting 3) start time rounding and end time rounding 4) use of buffering 5) frequency of run within the duration of event
# E.g. "/blizzard_allyrs_ceilfloor5min_nobuffer_freq5min.csv", "lakeeffect_2025_ceilfloor30min_nobuffer_freq30min"
d_event_subset.to_csv("/home/csutter/DRIVE-clean/NWS_warnings/data/ncei_events_ofinterest/heavyrain_2425_ceilfloor15min_nobuffer_freq15min.csv") #HERE!! 

# Important note: if you just click the CSV or even download the CSV and inspect in excel, it looks off... but if you read it in as a df and use it in a notebook, it reads in totally fine

In [68]:
# make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

datetimes_all = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        datetimes_all.append(j)

print(len(datetimes_all)) # this numnber should roughly match the # events x avg duration x numebr of runs per hour (e.g. 12 if running 5 min)
print(len(np.unique(datetimes_all)))

190
188


In [69]:
print(len(datetimes_all))
# Note that some will be overlapping for nearby counties or overlapping zones

dates_list = np.unique(datetimes_all)
print(len(dates_list)) # this is the amount of instances to run!

print(dates_list[0:4])

# Need to also see which datetimes I've already ran from past inference! See other notebook. 


190
188
['20240618_0900' '20240618_0915' '20240618_0930' '20240618_0945']


Grab list of datetimes already ran in inference 
- See here for the base reference code: /home/csutter/DRIVE-clean/operational_analysis/notebooks/summarize_dates_ran.ipynb

In [70]:
#HERE!! update this list

inf_sets_ran = ["/home/csutter/DRIVE-clean/operational_runs/set0_test",
"/home/csutter/DRIVE-clean/operational_runs/set1_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set2_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set3_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set4_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set5_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set6_probSRsample_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set7_probSRsample_20250926",
"/home/csutter/DRIVE-clean/operational_runs/set9_winter22_JFMOnly_000816",
"/home/csutter/DRIVE-clean/operational_runs/set10_winter2223_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set11_winer2324_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set12_winter2425_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set13_winter22_JFMOnly_041220",
"/home/csutter/DRIVE-clean/operational_runs/set14_winter2223_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set15_winter2324_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set16_winter2425_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set17_summer_examples",
"/home/csutter/DRIVE-clean/operational_runs/set18_summer_examples_2",
"/home/csutter/DRIVE-clean/operational_runs/set19_summer_examples_3",
"/home/csutter/DRIVE-clean/operational_runs/set20_summer_examples_4",
"/home/csutter/DRIVE-clean/operational_runs/set21_summer_examples_5",
"/home/csutter/DRIVE-clean/operational_runs/set22_summer_examples_6",
"/home/csutter/DRIVE-clean/operational_runs/set23_squall1",
"/home/csutter/DRIVE-clean/operational_runs/set24_squall2",
"/home/csutter/DRIVE-clean/operational_runs/set25_squall3",
"/home/csutter/DRIVE-clean/operational_runs/set26_blizzard1",
"/home/csutter/DRIVE-clean/operational_runs/set27_blizzard2",
"/home/csutter/DRIVE-clean/operational_runs/set28_blizzard3",
"/home/csutter/DRIVE-clean/operational_runs/set29_blizzard4",
"/home/csutter/DRIVE-clean/operational_runs/set30_blizzard5",
"/home/csutter/DRIVE-clean/operational_runs/set31_blizzard6",
"/home/csutter/DRIVE-clean/operational_runs/set32_blizzard7",
"/home/csutter/DRIVE-clean/operational_runs/set33_lakeeffect2025",
"/home/csutter/DRIVE-clean/operational_runs/set34_lakeeffect2025",
"/home/csutter/DRIVE-clean/operational_runs/set35_blizzard8_EgNoImgs"]

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


print(pred_datetimes[0:4])

6018
6018
['20250210_1000', '20250204_1000', '20250228_1000', '20250227_1000']


In [71]:
# Optional: if need to add in stuff that already IS running simulatneously so dont want to double on dates if they exist in multiple runs
# HERE!!

# OPT 1: if have 1 or 2 currently running jobs from which to remove dates running (then comment out OPT 2)

dates_runnING1 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/dates.csv")["date"])
print(len( dates_runnING1))
dates_runnING2 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set38_heavysnow/dates.csv")["date"])
dates_runnING3 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set39_winterstorm/dates.csv")["date"])
dates_runnING4 = list(pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set40_winterweather/dates.csv")["date"])
dates_runnING = dates_runnING1 + dates_runnING2 + dates_runnING3 + dates_runnING4
print(len(dates_runnING))

# OPT 2: if no currently running jobs to remove datetimes from list

# dates_runnING = []

2130
4942


In [72]:
# Cross check from dates_list and remove any dates already ran/running

dates_list_new = []
for d in dates_list:
    if d not in pred_datetimes:
        dates_list_new.append(d)

print("Unique datetimes from event of int")
print(len(dates_list))

print("Unique datetimes removing what's been ran")
print(len(dates_list_new))

dates_list_new2 = []
for d in dates_list_new:
    if d not in dates_runnING:
        dates_list_new2.append(d)

print("Unique datetimes removing what's currently running")
print(len(dates_list_new2))

Unique datetimes from event of int
188
Unique datetimes removing what's been ran
186
Unique datetimes removing what's currently running
186


In [64]:
# Save out 
forcsv = pd.DataFrame(dates_list_new2, columns=['date']) # ADJUST HERE!! Manually adjust how many instance to run in one sbatch run. Usually ~400 is good (takes 20 hours)
forcsv

savetodir = "/home/csutter/DRIVE-clean/operational_runs/set41_heavyrain" # ADJUST HERE!! Naming to match what event you're running
os.makedirs(savetodir, exist_ok=True)
forcsv.to_csv(f"{savetodir}/dates.csv") 